In [1]:
import sys
print(f"현재 사용 중인 파이썬 경로: {sys.executable}")

현재 사용 중인 파이썬 경로: /Users/sskim/anaconda3/envs/motornet/bin/python


In [5]:
!{sys.executable} -m pip install jPCA

In [17]:
# main_analysis.py

# 1. Import necessary libraries
import numpy as np
import scipy.io
import neo

import quantities as pq
import matplotlib.pyplot as plt
import elephant
from elephant.gpfa import GPFA
from jPCA.jPCA import JPCA
from jPCA.util import plot_projections


In [18]:

# 2. Data Loading and Preprocessing
def load_and_preprocess_data(filepath):
    """
    Loads the .mat file and converts the neural data into the
    neo.SpikeTrain format required by the elephant library.
    """
    # Load the .mat file
    data = scipy.io.loadmat(filepath)
    trial_data = data['trial']
    
    # Get dimensions
    n_trials, n_angles = trial_data.shape
    
    # This list will hold the data for all trials, for all angles
    all_trials_data = []
    
    # Loop through each reaching angle
    for angle_idx in range(n_angles):
        # This list will hold the data for all trials of a single angle
        angle_trials_data = []
        
        # Loop through each trial for the current angle
        for trial_idx in range(n_trials):
            # Get the spike data for the current trial
            spike_data = trial_data[trial_idx, angle_idx]['spikes']
            n_neurons, n_timesteps = spike_data.shape
            
            # This list will hold the SpikeTrain objects for each neuron in the trial
            trial_spiketrains = []
            
            # Loop through each neuron
            for neuron_idx in range(n_neurons):
                # Find the timestamps where a spike occurred
                spike_times = np.where(spike_data[neuron_idx, :] == 1)[0] * pq.ms
                
                # Create a neo.SpikeTrain object
                spiketrain = neo.SpikeTrain(spike_times, t_stop=n_timesteps * pq.ms)
                trial_spiketrains.append(spiketrain)
            
            angle_trials_data.append(trial_spiketrains)
        
        all_trials_data.append(angle_trials_data)
        
    return all_trials_data

# 3. Running GPFA
def run_gpfa(data, bin_size=20*pq.ms, x_dim=8):
    """
    Performs GPFA on the preprocessed neural data.
    """
    # Flatten the data list for GPFA input (it takes a list of trials)
    gpfa_input_data = [trial for angle_data in data for trial in angle_data]
    
    # Initialize GPFA
    gpfa = GPFA(bin_size=bin_size, x_dim=x_dim)
    
    # Fit and transform the data
    trajectories = gpfa.fit_transform(gpfa_input_data)
    
    return trajectories, gpfa

# 4. Plotting GPFA Trajectories
def plot_gpfa_trajectories(trajectories, n_trials_per_angle=100):
    """
    Plots the first 3 dimensions of the latent trajectories from GPFA.
    """
    plt.figure(figsize=(10, 8))
    ax = plt.axes(projection='3d')
    
    n_angles = len(trajectories) // n_trials_per_angle
    colors = plt.cm.jet(np.linspace(0, 1, n_angles))
    
    for i in range(len(trajectories)):
        angle_idx = i // n_trials_per_angle
        traj = trajectories[i]
        ax.plot(traj[0], traj[1], traj[2], color=colors[angle_idx], alpha=0.6)
        
    ax.set_xlabel('Latent Dim 1')
    ax.set_ylabel('Latent Dim 2')
    ax.set_zlabel('Latent Dim 3')
    ax.set_title('GPFA Latent Trajectories for Different Reaching Angles')
    plt.savefig('gpfa_trajectories.png')
    plt.show()

# 5. Running and Plotting jPCA
def run_and_plot_jpca(trajectories):
    """
    Performs jPCA on the GPFA trajectories and plots the results.
    """
    # jPCA requires a list of trajectories, where each is a (time, neurons) array
    # GPFA output is (neurons, time), so we need to transpose
    jpca_input = [traj.T for traj in trajectories]
    
    # Initialize and run jPCA
    jpca = JPCA(num_jpcs=2)
    jpca_results = jpca.fit(jpca_input)
    
    # Plot the jPCA projections
    plot_projections(jpca_results)
    plt.savefig('jpca_projections.png')
    plt.show()


In [20]:
data = scipy.io.loadmat('m1_reaching_data.mat')


In [35]:
# ===============================================================
# Part 0: Import Libraries
# ===============================================================
import numpy as np
import scipy.io
import neo
import quantities as pq
import matplotlib.pyplot as plt
from elephant.gpfa import GPFA

# ===============================================================
# Part 1: Final Preprocessing Function (IndexError Fixed)
# ===============================================================
def preprocess_reaching_data_final_v5(data, dataset_key='comboNjs'):
    """
    Final robust version that handles the shape difference caused by
    the `squeeze_me=True` option in scipy.io.loadmat.
    """
    print(f"--- Preprocessing '{dataset_key}' ---")
    
    data_struct_array = data[dataset_key]
    
    # --- FIX: Handle both 1D and 2D array shapes ---
    if data_struct_array.ndim == 1:
        # Shape is (n_neurons,) because of squeeze_me=True
        num_neurons = data_struct_array.shape[0]
    elif data_struct_array.ndim == 2:
        # Shape is (1, n_neurons)
        num_neurons = data_struct_array.shape[1]
    else:
        print(f"ERROR: Unexpected array dimension: {data_struct_array.ndim}")
        return None
        
    first_neuron_data = data_struct_array[0]
    num_conditions = len(first_neuron_data['cond'])
    
    print(f"Confirmed: {num_neurons} neurons and {num_conditions} conditions.")
    
    all_conditions_data = []
    for cond_idx in range(num_conditions):
        try:
            spikes_field = first_neuron_data['cond'][cond_idx]['spikes']
            num_trials, n_timesteps = spikes_field.shape
        except KeyError:
            print(f"\nFATAL ERROR: The struct at ['cond'][{cond_idx}] does NOT contain a 'spikes' field.")
            return None
        
        condition_trials = []
        for trial_idx in range(num_trials):
            trial_spiketrains = []
            for neuron_idx in range(num_neurons):
                neuron_struct = data_struct_array[neuron_idx]
                spikes_data = neuron_struct['cond'][cond_idx]['spikes'][trial_idx, :]
                spike_times = np.where(spikes_data == 1)[0] * pq.ms
                spiketrain = neo.SpikeTrain(spike_times, t_stop=n_timesteps * pq.ms)
                trial_spiketrains.append(spiketrain)
            condition_trials.append(trial_spiketrains)
            
        all_conditions_data.append(condition_trials)
        print(f"  - Condition {cond_idx}: Processed {num_trials} trials.")
        
    print("--- Preprocessing Complete ---")
    return all_conditions_data

# ===============================================================
# Part 2: GPFA and Plotting (No changes needed)
# ===============================================================
def run_and_plot_gpfa(processed_data, bin_size=20*pq.ms, x_dim=8):
    if not processed_data:
        print("GPFA cancelled due to preprocessing failure.")
        return

    all_trials = [trial for condition_list in processed_data for trial in condition_list]
    
    print(f"\n--- Running GPFA on {len(all_trials)} total trials ---")
    gpfa = GPFA(bin_size=bin_size, x_dim=x_dim)
    trajectories = gpfa.fit_transform(all_trials)
    print("--- GPFA Complete ---")

    # Plotting
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    num_conditions = len(processed_data)
    colors = plt.cm.jet(np.linspace(0, 1, num_conditions))
    
    trial_counter = 0
    for cond_idx, condition_trials in enumerate(processed_data):
        for _ in condition_trials:
            traj = trajectories[trial_counter]
            ax.plot(traj[0, :], traj[1, :], traj[2, :], color=colors[cond_idx], alpha=0.4)
            trial_counter += 1
    
    ax.set_xlabel('Latent Dimension 1')
    ax.set_ylabel('Latent Dimension 2')
    ax.set_zlabel('Latent Dimension 3')
    ax.set_title(f'GPFA Latent Trajectories (Latent Dim = {x_dim})')
    plt.show()

# ===============================================================
# Part 3: Main Execution Block
# ===============================================================
    

In [ ]:
data_filepath = 'm1_reaching_data.mat'
print(f"Loading data from '{data_filepath}'...")
mat_data = scipy.io.loadmat(data_filepath, squeeze_me=True)
print("File loaded successfully.")


Loading data from 'm1_reaching_data.mat'...
File loaded successfully.
An unexpected error occurred: name 'preprocess_reaching_data_final_v5' is not defined


In [37]:
# 'cond' 구조체 안에 있는 최종 필드 이름들을 확인합니다.
# squeeze_me=True 옵션을 사용했으므로 인덱싱이 간단해졌습니다.
print(mat_data['comboNjs'][0]['cond'][0].dtype.names)

('mazeID', 'trialVersion', 'protoTrial', 'TargLocked', 'GoLocked', 'MoveLocked', 'interpPSTH')


In [ ]:
processed_data = preprocess_reaching_data_final_v5(mat_data, dataset_key='comboNjs')

run_and_plot_gpfa(processed_data, bin_size=20*pq.ms, x_dim=8)


--- Preprocessing 'comboNjs' ---
Confirmed: 161 neurons and 27 conditions.


ValueError: no field of name spikes

In [33]:
data_filepath = 'm1_reaching_data.mat'
    
print(f"Loading data from '{data_filepath}'...")
# Using squeeze_me=True simplifies array indexing
mat_data = scipy.io.loadmat(data_filepath, squeeze_me=True)
print("File loaded successfully.")

# Analyze the 'comboNjs' dataset
processed_data = preprocess_reaching_data_final(mat_data, dataset_key='comboNjs')

Loading data from 'm1_reaching_data.mat'...
File loaded successfully.
--- Preprocessing 'comboNjs' ---


IndexError: tuple index out of range

In [ ]:
try:
    print(f"Loading data from '{data_filepath}'...")
    # Using squeeze_me=True simplifies array indexing
    mat_data = scipy.io.loadmat(data_filepath, squeeze_me=True)
    print("File loaded successfully.")

    # Analyze the 'comboNjs' dataset
    processed_data = preprocess_reaching_data_final(mat_data, dataset_key='comboNjs')
    
    # If preprocessing was successful, run the analysis
    if processed_data:
        run_and_plot_gpfa(processed_data, bin_size=20*pq.ms, x_dim=8)

except FileNotFoundError:
    print(f"ERROR: Data file not found at '{data_filepath}'.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [11]:

# Main execution block
# Define the path to your data file
data_filepath = 'monkeydata.mat'

# 1. Load and preprocess the data
print("Loading and preprocessing data...")
all_trials_data = load_and_preprocess_data(data_filepath)
print("Data preprocessing complete.")


Loading and preprocessing data...
Data preprocessing complete.


In [19]:

# 2. Run GPFA
# We will use data from all angles for fitting the GPFA model
print("Running GPFA...")
trajectories, gpfa_model = run_gpfa(all_trials_data, bin_size=20*pq.ms, x_dim=8)
print("GPFA complete.")


Running GPFA...


/Users/sskim/anaconda3/envs/motornet/lib/python3.13/site-packages/elephant/conversion.py:1130: UserWarning: Binning discarded 15 last spike(s) of the input spiketrain
  warnings.warn("Binning discarded {} last spike(s) of the "
/Users/sskim/anaconda3/envs/motornet/lib/python3.13/site-packages/elephant/conversion.py:1130: UserWarning: Binning discarded 6 last spike(s) of the input spiketrain
  warnings.warn("Binning discarded {} last spike(s) of the "
/Users/sskim/anaconda3/envs/motornet/lib/python3.13/site-packages/elephant/conversion.py:1130: UserWarning: Binning discarded 12 last spike(s) of the input spiketrain
  warnings.warn("Binning discarded {} last spike(s) of the "
/Users/sskim/anaconda3/envs/motornet/lib/python3.13/site-packages/elephant/conversion.py:1130: UserWarning: Binning discarded 19 last spike(s) of the input spiketrain
  warnings.warn("Binning discarded {} last spike(s) of the "
/Users/sskim/anaconda3/envs/motornet/lib/python3.13/site-packages/elephant/conversion.py:

ValueError: Observation covariance matrix is rank deficient.
Possible causes: repeated units, not enough observations.

In [ ]:

# 3. Plot GPFA trajectories
print("Plotting GPFA trajectories...")
plot_gpfa_trajectories(trajectories, n_trials_per_angle=100)
print("GPFA plot saved as gpfa_trajectories.png")

# 4. Run and plot jPCA
print("Running jPCA...")
run_and_plot_jpca(trajectories)
print("jPCA plot saved as jpca_projections.png")